In [5]:
#эту ячейку выполнил Ринат
import threading
import time

TOTAL_SEATS = 100
CASHIERS = 8

seats = [False] * TOTAL_SEATS
sold_tickets = []

def cashier(cashier_id):
    while True:
        free_seat = None

        for index in range(TOTAL_SEATS):
            if not seats[index]:
                free_seat = index
                break

        if free_seat is None:
            return

        # Небольшая пауза позволяет проявиться гонке потоков.
        time.sleep(0.0001)

        seats[free_seat] = True
        sold_tickets.append({
            "seat": free_seat + 1,
            "cashier": cashier_id
        })

start = time.perf_counter()

threads = []
for cashier_id in range(1, CASHIERS + 1):
    thread = threading.Thread(
        target=cashier,
        args=(cashier_id,)
    )
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

elapsed = time.perf_counter() - start

unique_seats = len({ticket["seat"] for ticket in sold_tickets})

print("Продано билетов:", len(sold_tickets))
print("Уникальных мест:", unique_seats)
print(f"Время выполнения: {elapsed:.4f} сек.")

Продано билетов: 710
Уникальных мест: 100
Время выполнения: 0.0212 сек.


**Race Condition**

1. Касса № 1 проверяет место и видит, что оно свободно.
2. До того как касса № 1 отметит место занятым, касса № 2 проверяет то же место.
3. Касса № 2 тоже считает место свободным.
4. Обе кассы регистрируют продажу одного места.

Таким образом, без синхронизации несколько потоков могут одновременно изменять общие данные, что приводит к повторной продаже билетов.

In [2]:
#эту ячейку выполнил Ринат
import threading
import time

TOTAL_SEATS = 100
CASHIERS = 8

seats = [False] * TOTAL_SEATS
sold_tickets = []
lock = threading.Lock()

def cashier(cashier_id):
    while True:
        with lock:
            free_seat = None

            for index in range(TOTAL_SEATS):
                if not seats[index]:
                    free_seat = index
                    break

            if free_seat is None:
                return

            seats[free_seat] = True
            sold_tickets.append({
                "seat": free_seat + 1,
                "cashier": cashier_id
            })

        # Печать выполняется после освобождения блокировки.
        print(f"Касса {cashier_id}: место {free_seat + 1}")

start = time.perf_counter()

threads = []
for cashier_id in range(1, CASHIERS + 1):
    thread = threading.Thread(
        target=cashier,
        args=(cashier_id,)
    )
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

elapsed = time.perf_counter() - start

unique_seats = len({ticket["seat"] for ticket in sold_tickets})

print("\n--- Итоги ---")
print("Количество продаж:", len(sold_tickets))
print("Уникальных мест:", unique_seats)
print(f"Время выполнения: {elapsed:.4f} сек.")

assert len(sold_tickets) == 100
assert unique_seats == 100

print("Проверка пройдена: все 100 мест проданы по одному разу.")

Касса 1: место 1Касса 2: место 2
Касса 2: место 3
Касса 2: место 4
Касса 2: место 5
Касса 2: место 6
Касса 2: место 7
Касса 2: место 8
Касса 2: место 9
Касса 2: место 10
Касса 2: место 11
Касса 2: место 12
Касса 2: место 13
Касса 2: место 14
Касса 2: место 15
Касса 2: место 16
Касса 2: место 17
Касса 2: место 18
Касса 2: место 19
Касса 2: место 20
Касса 2: место 21
Касса 2: место 22
Касса 2: место 23
Касса 2: место 24
Касса 2: место 25
Касса 2: место 26
Касса 2: место 27
Касса 2: место 28
Касса 2: место 29
Касса 2: место 30
Касса 2: место 31
Касса 2: место 32
Касса 2: место 33
Касса 2: место 34
Касса 2: место 35
Касса 2: место 36
Касса 2: место 37
Касса 2: место 38
Касса 2: место 39
Касса 2: место 40
Касса 2: место 41
Касса 2: место 42
Касса 2: место 43
Касса 2: место 44
Касса 2: место 45
Касса 2: место 46
Касса 2: место 47
Касса 2: место 48
Касса 2: место 49
Касса 2: место 50
Касса 2: место 51
Касса 2: место 52
Касса 2: место 53
Касса 2: место 54
Касса 2: место 55
Касса 2: место 56
Ка

In [3]:
#эту ячейку выполнил Мирсултан
import threading
import time

def run_test(print_inside_lock):
    seats = [False] * 100
    lock = threading.Lock()

    def cashier():
        while True:
            with lock:
                free_seat = next(
                    (i for i, occupied in enumerate(seats)
                     if not occupied),
                    None
                )

                if free_seat is None:
                    return

                seats[free_seat] = True

                if print_inside_lock:
                    print(f"Продано место {free_seat + 1}")

            if not print_inside_lock:
                print(f"Продано место {free_seat + 1}")

    start = time.perf_counter()

    threads = [
        threading.Thread(target=cashier)
        for _ in range(8)
    ]

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    return time.perf_counter() - start

inside = run_test(True)
outside = run_test(False)

print("\n--- Сравнение ---")
print(f"Печать внутри блокировки: {inside:.4f} сек.")
print(f"Печать вне блокировки:    {outside:.4f} сек.")

Продано место 1
Продано место 2
Продано место 3
Продано место 4
Продано место 5
Продано место 6
Продано место 7
Продано место 8
Продано место 9
Продано место 10
Продано место 11
Продано место 12
Продано место 13
Продано место 14
Продано место 15
Продано место 16
Продано место 17
Продано место 18
Продано место 19
Продано место 20
Продано место 21
Продано место 22
Продано место 23
Продано место 24
Продано место 25
Продано место 26
Продано место 27
Продано место 28
Продано место 29
Продано место 30
Продано место 31
Продано место 32
Продано место 33
Продано место 34
Продано место 35
Продано место 36
Продано место 37
Продано место 38
Продано место 39
Продано место 40
Продано место 41
Продано место 42
Продано место 43
Продано место 44
Продано место 45
Продано место 46
Продано место 47
Продано место 48
Продано место 49
Продано место 50
Продано место 51
Продано место 52
Продано место 53
Продано место 54
Продано место 55
Продано место 56
Продано место 57
Продано место 58
Продано место 59
Продан